<a href="https://colab.research.google.com/github/springboardmentor123g/PlantDocBot/blob/intern-AnshikaSahu/Img_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import kagglehub
import os, json, shutil

path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
dataset_path = os.path.join(path, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)")
train_path = os.path.join(dataset_path, "train")
valid_path = os.path.join(dataset_path, "valid")

transform_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor()
])

transform_valid = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(root=train_path, transform=transform_train)
valid_dataset = datasets.ImageFolder(root=valid_path, transform=transform_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
os.makedirs("model_folder", exist_ok=True)
with open("model_folder/class_mapping.json", "w") as f:
    json.dump(idx_to_class, f)


In [ ]:
class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.classes)

model = PlantCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0008)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.7)

epochs = 13

for epoch in range(epochs):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    train_acc = 100 * correct / total
    scheduler.step()

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    val_acc = 100 * val_correct / val_total
    print(f"Epoch [{epoch+1}/{epochs}] Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from google.colab import drive
drive.flush_and_unmount()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
torch.save(model.state_dict(), "model_folder/plant_cnn.pth")

drive_path = "/content/drive/MyDrive/plant_disease_model/"
os.makedirs(drive_path, exist_ok=True)
shutil.copy("model_folder/plant_cnn.pth", drive_path)
shutil.copy("model_folder/class_mapping.json", drive_path)
print("Saved model and mapping to Drive")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image


In [ ]:
class PlantCNN(torch.nn.Module):
    def __init__(self, num_classes):
        super(PlantCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 32, 3, 1, 1)
        self.bn1 = torch.nn.BatchNorm2d(32)
        self.conv2 = torch.nn.Conv2d(32, 64, 3, 1, 1)
        self.bn2 = torch.nn.BatchNorm2d(64)
        self.conv3 = torch.nn.Conv2d(64, 128, 3, 1, 1)
        self.bn3 = torch.nn.BatchNorm2d(128)
        self.conv4 = torch.nn.Conv2d(128, 256, 3, 1, 1)
        self.bn4 = torch.nn.BatchNorm2d(256)
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.dropout = torch.nn.Dropout(0.5)
        self.fc1 = torch.nn.Linear(256 * 8 * 8, 512)
        self.fc2 = torch.nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


In [ ]:
drive_path = "/content/drive/MyDrive/plant_disease_model"
model_path = os.path.join(drive_path, "plant_cnn.pth")
mapping_path = os.path.join(drive_path, "class_mapping.json")

with open(mapping_path, "r") as f:
    idx_to_class = json.load(f)

num_classes = len(idx_to_class)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PlantCNN(num_classes).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model and mapping loaded. Device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

def predict_image_with_confidence(image_path):
    image = Image.open(image_path).convert("RGB")
    img_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = F.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probs, 1)
        predicted_class = idx_to_class[str(predicted.item())]
    return predicted_class, confidence.item()


image_path = "/content/drive/MyDrive/test_img2.jpg"
pred_class, confidence = predict_image_with_confidence(image_path)
print("Predicted disease:", pred_class)
print(f"Confidence: {confidence*100:.2f}%")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import json
import os

# ----------------- Load mapping & model -----------------
drive_path = "/content/drive/MyDrive/plant_disease_model/"
model_path = os.path.join(drive_path, "plant_cnn.pth")
mapping_path = os.path.join(drive_path, "class_mapping.json")

with open(mapping_path, "r") as f:
    idx_to_class = json.load(f)

num_classes = len(idx_to_class)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------- Model Architecture -----------------
class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = PlantCNN(num_classes).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# ----------------- Test-Time Augmentation -----------------
tta_transforms = [
    transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
]

def predict_with_tta(image_path):
    image = Image.open(image_path).convert("RGB")
    probs_total = torch.zeros(num_classes).to(device)

    for transform in tta_transforms:
        img_tensor = transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(img_tensor)
            probs = F.softmax(outputs, dim=1).squeeze(0)
            probs_total += probs

    probs_avg = probs_total / len(tta_transforms)
    top2_conf, top2_idx = torch.topk(probs_avg, 2)

    results = [(idx_to_class[str(idx.item())], conf.item()) for idx, conf in zip(top2_idx, top2_conf)]
    return results

# ----------------- Example Usage -----------------
image_path = "/content/drive/MyDrive/test_img2.jpg"  # Update your image path
predictions = predict_with_tta(image_path)

print("Top-2 Predictions with confidence:")
for cls, conf in predictions:
    print(f"{cls}: {conf*100:.2f}%")
